# Initialize Unity Catalog

Creates the verdict catalog, schemas, and Delta tables

In [ ]:
import sys, os
notebook_dir = os.getcwd()
src_path = os.path.join(os.path.dirname(notebook_dir), 'src')
if src_path not in sys.path: sys.path.insert(0, src_path)

import logging
logging.basicConfig(level=logging.INFO)

from verdict.setup.init_catalog import VerdictCatalogSetup
from verdict.orchestration import RunContext

In [ ]:
# Get run context (pipeline or standalone)
dbutils.widgets.text("run_id", "", "Run ID")
dbutils.widgets.text("catalog_name", "verdict_dev", "Catalog Name")

run_id = dbutils.widgets.get("run_id") or None
catalog_name = dbutils.widgets.get("catalog_name")

ctx = RunContext.from_run_id(spark, catalog_name, run_id) if run_id else None
config = ctx.get_config() if ctx else {}
catalog_name = config.get("catalog_name", catalog_name)

In [ ]:
if ctx:
    ctx.update_task_status("init_catalog", "running")

In [ ]:
logging.getLogger(__name__).info(f"Initializing Unity Catalog: {catalog_name}")
VerdictCatalogSetup(catalog_name=catalog_name).run_setup()

In [ ]:
if ctx:
    ctx.update_task_status("init_catalog", "completed")

In [ ]:
# Verify tables
for schema in ['raw', 'evaluated', 'metrics', 'metadata']:
    spark.sql(f"SHOW TABLES IN {catalog_name}.{schema}").display()